# A/B-тест email-рассылки (Hillstrom): анализ и uplift-модель

**Часть 1: данные, дизайн теста, A/B-анализ.**

Контекст: интернет-магазин разослал клиентам письма двух видов (мужская и женская подборка) и оставил контрольную группу без письма.
Нужно ответить на вопрос бизнеса: **стоит ли отправлять письма и какое именно?**

План ноутбука:
1. Знакомство с данными и проверка качества
2. Дизайн теста: гипотезы, метрики, SRM, баланс групп, мощность
3. A/B-анализ: конверсия, визиты, траты, выводы

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint, proportion_effectsize
from statsmodels.stats.power import NormalIndPower

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Путь находит файл и на Kaggle, и локально
paths = glob.glob("/kaggle/input/**/*Hillstrom*.csv", recursive=True) \
     or glob.glob("/kaggle/input/**/*.csv", recursive=True) \
     or glob.glob("../archive/*.csv")
df = pd.read_csv(paths[0])
print("Файл:", paths[0])
df.shape

## 1. Знакомство с данными

Колонки:
- `recency` — месяцев с последней покупки; `history`, `history_segment` — сумма покупок за прошлый год
- `mens`, `womens` — покупал ли мужские или женские товары
- `zip_code` — тип района; `newbie` — новый клиент (1) или нет (0); `channel` — канал покупок
- `segment` — **группа теста**: `Mens E-Mail`, `Womens E-Mail`, `No E-Mail` (контроль)
- `visit`, `conversion`, `spend` — результаты за две недели после рассылки

In [ ]:
display(df.head())
display(df.dtypes.to_frame("dtype").T)
print("Пропуски:", int(df.isna().sum().sum()))
print("Полных дублей строк:", int(df.duplicated().sum()))

Полные дубли строк здесь ничего не значат: в данных нет id клиента, поэтому два разных клиента могут иметь одинаковые значения признаков.
Поэтому дубли не удаляем.

In [ ]:
display(df.describe().T)

# Целевые метрики: проверяем, что значения корректны
assert df["visit"].isin([0, 1]).all() and df["conversion"].isin([0, 1]).all()
assert (df["spend"] >= 0).all()
# Покупка без визита была бы нелогична
print("Конверсий без визита:", int(((df.conversion == 1) & (df.visit == 0)).sum()))
print("Трата > 0 без конверсии:", int(((df.spend > 0) & (df.conversion == 0)).sum()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["history"], bins=50, ax=axes[0]).set(title="Сумма прошлых покупок")
sns.histplot(df.loc[df.spend > 0, "spend"], bins=40, ax=axes[1]).set(title="Траты (только покупатели)")
plt.tight_layout(); plt.show()

Траты сильно скошены вправо (много маленьких покупок, мало больших), а покупателей мало. Это важно для выбора метода: для трат кроме t-теста возьмём ещё бутстрап.

## 2. Дизайн теста

### Гипотезы
Сравниваем каждое письмо с контролем (`No E-Mail`).

- **H0:** письмо не влияет на конверсию в покупку.
- **H1:** письмо меняет конверсию (двусторонняя проверка).

### Метрики
| Тип | Метрика | Почему |
|---|---|---|
| Основная | `conversion` (доля купивших) | Прямая бизнес-цель рассылки |
| Вторичные | `visit` (доля зашедших), `spend` (средняя сумма на клиента) | Показывают механизм эффекта и деньги |

### Уровень значимости
Сравнений с контролем два, поэтому применяем **поправку Бонферрони**: α = 0,05 / 2 = 0,025.

In [ ]:
ALPHA = 0.05
N_COMPARISONS = 2
ALPHA_ADJ = ALPHA / N_COMPARISONS

groups = ["No E-Mail", "Mens E-Mail", "Womens E-Mail"]
sizes = df["segment"].value_counts().reindex(groups)
display(sizes.to_frame("n"))

### Проверка SRM (Sample Ratio Mismatch)
Если рассылка делилась случайно на три равные части, в каждой группе должно быть примерно по 1/3 клиентов. Проверяем хи-квадратом.
Маленький p-value значил бы, что распределение нарушено и результатам теста доверять нельзя.

In [ ]:
expected = np.repeat(len(df) / 3, 3)
chi2, p_srm = stats.chisquare(sizes.values, expected)
print(f"SRM: chi2 = {chi2:.3f}, p-value = {p_srm:.4f}")
print("Вывод:", "нарушение деления групп" if p_srm < 0.001 else "деление групп в порядке")

### Баланс групп по признакам
При случайном делении признаки клиентов должны быть распределены одинаково во всех группах. Проверяем это.

In [ ]:
cat_cols = ["newbie", "mens", "womens", "zip_code", "channel", "history_segment"]
rows = []
for c in cat_cols:
    chi2_c, p_c, _, _ = stats.chi2_contingency(pd.crosstab(df["segment"], df[c]))
    rows.append((c, p_c))
for c in ["recency", "history"]:
    f, p_c = stats.f_oneway(*[df.loc[df.segment == g, c] for g in groups])
    rows.append((c, p_c))
balance = pd.DataFrame(rows, columns=["признак", "p-value"])
balance["баланс"] = np.where(balance["p-value"] > 0.05, "ок", "проверить")
display(balance)

Если p-value больше 0,05 для всех признаков, группы сопоставимы и различия в результатах можно приписать письму, а не составу групп.
Один-два «проверить» при 8 признаках могут появиться случайно: при α = 0,05 в среднем 1 из 20 проверок срабатывает впустую.

### Мощность и минимальный обнаруживаемый эффект (MDE)
Какой минимальный рост конверсии тест мог надёжно заметить при таких размерах групп?

In [ ]:
ctrl = df[df.segment == "No E-Mail"]
n_ctrl = len(ctrl)
p0 = ctrl["conversion"].mean()
print(f"Базовая конверсия контроля: {p0:.4%}, n на группу ≈ {n_ctrl}")

analysis = NormalIndPower()
lifts = np.linspace(0.05, 1.0, 96)  # относительный рост конверсии
power = [analysis.power(proportion_effectsize(p0 * (1 + l), p0), nobs1=n_ctrl, alpha=ALPHA_ADJ, ratio=1) for l in lifts]

mde_rel = next(l for l, pw in zip(lifts, power) if pw >= 0.8)
print(f"MDE при мощности 80% и α = {ALPHA_ADJ}: рост конверсии на {mde_rel:.0%} относительно базы "
      f"({p0:.3%} → {p0 * (1 + mde_rel):.3%})")

plt.figure(figsize=(7, 4))
plt.plot(lifts * 100, power)
plt.axhline(0.8, ls="--", c="gray")
plt.axvline(mde_rel * 100, ls="--", c="gray")
plt.xlabel("Относительный рост конверсии, %"); plt.ylabel("Мощность"); plt.title("Мощность теста")
plt.show()

Конверсия в данных низкая (меньше 1%), поэтому для надёжного обнаружения нужен заметный относительный рост. Это ограничение стоит помнить, когда будем читать результат: «нет значимого эффекта» тут не значит «эффекта нет».

## 3. A/B-анализ

In [ ]:
def summarize(group):
    d = df[df.segment == group]
    n = len(d)
    return pd.Series({
        "n": n,
        "visit_rate": d["visit"].mean(),
        "conversion_rate": d["conversion"].mean(),
        "spend_per_user": d["spend"].mean(),
    })

summary = pd.DataFrame({g: summarize(g) for g in groups}).T
display(summary)

### Конверсия и визиты: z-тест для долей
Для каждого письма считаем разницу с контролем, 95%-й доверительный интервал разницы и p-value. Решение принимаем по скорректированному α.

In [ ]:
def prop_test(metric, treat, control="No E-Mail"):
    t = df[df.segment == treat][metric]
    c = df[df.segment == control][metric]
    count = np.array([t.sum(), c.sum()])
    nobs = np.array([len(t), len(c)])
    _, p = proportions_ztest(count, nobs)
    pt, pc = t.mean(), c.mean()
    diff = pt - pc
    se = np.sqrt(pt * (1 - pt) / len(t) + pc * (1 - pc) / len(c))
    ci_low, ci_high = diff - 1.96 * se, diff + 1.96 * se
    return {
        "метрика": metric, "группа": treat,
        "контроль": pc, "тест": pt,
        "разница (п.п.)": diff * 100, "CI95 низ (п.п.)": ci_low * 100, "CI95 верх (п.п.)": ci_high * 100,
        "относит. рост": diff / pc, "p-value": p,
        "значимо (Бонферрони)": p < ALPHA_ADJ,
    }

res_prop = pd.DataFrame([prop_test(m, g) for m in ["conversion", "visit"] for g in ["Mens E-Mail", "Womens E-Mail"]])
display(res_prop)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["conversion", "visit"]):
    rates, errs = [], []
    for g in groups:
        d = df[df.segment == g][metric]
        lo, hi = proportion_confint(d.sum(), len(d), method="wilson")
        rates.append(d.mean() * 100)
        errs.append([d.mean() * 100 - lo * 100, hi * 100 - d.mean() * 100])
    ax.bar(groups, rates, yerr=np.array(errs).T, capsize=6, color=["#999", "#4c78a8", "#e45756"])
    ax.set_title(f"{metric}, % (95% CI)")
plt.tight_layout(); plt.show()

### Траты: t-тест Уэлча и бутстрап
Траты скошены и в основном состоят из нулей, поэтому проверяем двумя способами: классическим t-тестом и бутстрапом разницы средних (он не требует нормальности).

In [ ]:
rng = np.random.default_rng(42)

def spend_test(treat, control="No E-Mail", n_boot=5000):
    t = df[df.segment == treat]["spend"].to_numpy()
    c = df[df.segment == control]["spend"].to_numpy()
    _, p = stats.ttest_ind(t, c, equal_var=False)
    boots = np.empty(n_boot)
    for i in range(n_boot):
        boots[i] = rng.choice(t, len(t)).mean() - rng.choice(c, len(c)).mean()
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {
        "группа": treat, "контроль $": c.mean(), "тест $": t.mean(),
        "разница $": t.mean() - c.mean(), "бутстрап CI95 низ": lo, "бутстрап CI95 верх": hi,
        "p-value (Уэлч)": p, "значимо (Бонферрони)": p < ALPHA_ADJ,
    }

res_spend = pd.DataFrame([spend_test(g) for g in ["Mens E-Mail", "Womens E-Mail"]])
display(res_spend)

### Mens против Womens
Какое письмо лучше? Прямое сравнение между двумя письмами. Это третье сравнение, поэтому не забываем про множественность.

In [ ]:
res_mw = pd.DataFrame([prop_test(m, "Mens E-Mail", control="Womens E-Mail") for m in ["conversion", "visit"]])
display(res_mw)

## Промежуточные выводы (заполним после запуска на Kaggle)

- Письма увеличивают визиты и конверсию? Какое из двух сильнее?
- Покрывает ли доверительный интервал нулевой эффект?
- Окупается ли рассылка, если известна стоимость отправки одного письма?
- Что показал анализ мощности: какие эффекты тест мог не заметить?

Дальше: сегментный анализ (где эффект сильнее и где он отрицательный), затем uplift-модель.